In [131]:
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split


Load Data

In [132]:
data = datasets.load_iris()

In [133]:
X = data["data"]
y = data["target"]

In [134]:
y = np.eye(3, dtype=float)[y] # one hot encoding for the class

In [135]:
X.shape, y.shape

((150, 4), (150, 3))

Train-test split

In [136]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42     # makes the split reproducible
)


In [137]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((120, 4), (30, 4), (120, 3), (30, 3))

Create functions required by the neural network

In [138]:
# Parameters of the neural network, these are configurable
# for now we will hard code a few things, and then find a way to make this more configurable easily

# Only this needs to be changed if we are changing the architecture, assuming a 3 layer network
NEURONS_INFO = {0: 8, 1: 4, 2: 3} # E.g., {0: 3, 1: 2} -> first layer has 3 neurons, second layer has 2 neurons
N_LAYERS = len(NEURONS_INFO.keys())


In [139]:
# Change the number of neurons in each layer to the maximum `m`, from the `m_l` neurons for each layer `l`
# Note that now the weights and biases for those inactive nodes must be initialised to zero

M = -1 # max. no of neurons in a layer
for m_l in NEURONS_INFO.values():
    M = max(m_l, M)
    

In [140]:
m1 = [np.random.uniform(-1,1,(X_train.shape[1], NEURONS_INFO[0])),
                    np.random.uniform(-1,1,(NEURONS_INFO[0], NEURONS_INFO[1])),
                    np.random.uniform(-1,1,(NEURONS_INFO[1], NEURONS_INFO[2]))]
m2 = [np.random.uniform(-1,1,(NEURONS_INFO[0],1)),
                np.random.uniform(-1,1,(NEURONS_INFO[1],1)),
                np.random.uniform(-1,1,(NEURONS_INFO[2],1))]

In [141]:
m1[0].shape, m1[1].shape

((4, 8), (8, 4))

In [142]:
def format_matrices(matrices, matrix_type):
    """
    Given a list of numpy matrices, change the values of the inactive values to zero 
    matrix_type could be "weights" or "biases"
    """
    updated_matrices = []
    
    if matrix_type == "weights":
        for i, matrix_i in enumerate(matrices):
            row_pad_shape = (0, 0)
            col_pad_shape = (0, 0)
            if i == 0:
                col_pad_shape = (0,  M - matrix_i.shape[1])
            else:
                row_pad_shape = (0, M - matrix_i.shape[0])
                col_pad_shape = (0, M - matrix_i.shape[1])
                
            matrix_i = np.pad(matrix_i,
            (row_pad_shape,
            col_pad_shape),
            mode="constant")
            updated_matrices.append(matrix_i)
    elif matrix_type == "biases":
                for i, matrix_i in enumerate(matrices):
                    row_pad_shape = (0, M - matrix_i.shape[0])
                    matrix_i = np.pad(matrix_i,
                    (row_pad_shape,
                    (0,0)),
                    mode="constant")
                    updated_matrices.append(matrix_i)
    else:
        raise ValueError("Invalid matrix type")
    
    return updated_matrices  
    

In [143]:
# format_matrices(m1, "weights")

In [144]:
# format_matrices(m2, "biases")

In [145]:
def sigmoid(x):
    return 1/(1+np.exp(-x))

In [146]:
def forward_pass(row, weights_matrices, bias_matrices):
    """Given a single row of training data,
    this function calculates the output produced by the network.
    
    The dimension of `row` is (<no. of features>, ).
    """
    row = row.reshape((1,-1))
    z_values = [0]*N_LAYERS
    a_values = [0]*N_LAYERS
    
    for layer in range(N_LAYERS):
        if layer == 0:
            # print(f"Layer {layer}: σ(X {row.shape} x W {weights_matrices[layer].shape} + B {bias_matrices[layer].T.shape}")
            z_values[layer] = row @ weights_matrices[layer] + bias_matrices[layer].T
        else:
            z_values[layer] = a_values[layer-1] @ weights_matrices[layer] + bias_matrices[layer].T
            # print(f"Layer {layer}: σ(A {a_values[layer-1].shape} x W {weights_matrices[layer].shape} + B {bias_matrices[layer].T.shape}")
        a_values[layer] = sigmoid(z_values[layer])
        

    return z_values, a_values

In [147]:
def train(X_train, y_train, epochs = 10, eta = 0.01):
    
    training_size = X_train.shape[0]
    
    # Initialise weights and biases
    np.random.seed(42)
    weights_matrices = format_matrices([np.random.uniform(-1,1,(X_train.shape[1], NEURONS_INFO[0])),
                        np.random.uniform(-1,1,(NEURONS_INFO[0], NEURONS_INFO[1])),
                        np.random.uniform(-1,1,(NEURONS_INFO[1], NEURONS_INFO[2]))], "weights")
    bias_matrices = format_matrices([np.random.uniform(-1,1,(NEURONS_INFO[0],1)),
                    np.random.uniform(-1,1,(NEURONS_INFO[1],1)),
                    np.random.uniform(-1,1,(NEURONS_INFO[2],1))], "biases")
    
    for epoch in range(epochs):
        
        # Initialise loss and gradient values
        loss = 0 # this loss is for the entire training dataset (which is our batch)
        
        dC_dz = [np.zeros((NEURONS_INFO[0],1)),
                np.zeros((NEURONS_INFO[1],1)),
                np.zeros((NEURONS_INFO[2],1))]
        dC_dw = [np.zeros((X_train.shape[1], NEURONS_INFO[0])),
                np.zeros((NEURONS_INFO[0], NEURONS_INFO[1])),
                np.zeros((NEURONS_INFO[1], NEURONS_INFO[2]))]
        dC_db = [np.zeros((NEURONS_INFO[0],1)),
                np.zeros((NEURONS_INFO[1],1)),
                np.zeros((NEURONS_INFO[2],1))]
        
        for X_i, y_i in zip(X_train, y_train):
                # Get all linear and activated outputs via the forward pass
                z_values_i, a_values_i = forward_pass(X_i, weights_matrices, bias_matrices)

                # Initialising gradients for the current training sample
                dC_dz_i = [np.zeros((NEURONS_INFO[0],1)),
                        np.zeros((NEURONS_INFO[1],1)),
                        np.zeros((NEURONS_INFO[2],1))]
                dC_dw_i = [np.zeros((X_train.shape[1], NEURONS_INFO[0])),
                        np.zeros((NEURONS_INFO[0], NEURONS_INFO[1])),
                        np.zeros((NEURONS_INFO[1], NEURONS_INFO[2]))]
                dC_db_i = [np.zeros((NEURONS_INFO[0],1)),
                        np.zeros((NEURONS_INFO[1],1)),
                        np.zeros((NEURONS_INFO[2],1))]
            
                
                #     # Calculating dC_dz_i for last layer, *can this be done for each layer via a loop in a generic way?
                for layer in range(N_LAYERS-1, -1, -1): # N_LAYERS-1 is inclusive, -1 is exclusive, -1 is the step
                        for neuron_j in range(NEURONS_INFO[layer]): # neuron 'j' of layer 'layer'
                                a_layer_vals= a_values_i[layer].reshape(-1) # from shape (1, <no of neurons in layer>) to (<no of neurons in layer>) to help calculations below
                                a_layer_vals = a_layer_vals[:-(M - NEURONS_INFO[layer])] # remove the inactive values
                                if layer == N_LAYERS-1: # last layer
                                        dC_dz_i[layer][neuron_j] = 2 * (a_layer_vals[neuron_j] - y_i[neuron_j]) * (a_layer_vals[neuron_j] * (1-a_layer_vals[neuron_j]))
                                # else:
                                        
                                #         dC_dz_i[layer][neuron_j]  = 

                    
                    
            
                
                
                last_layer_idx = N_LAYERS - 1 
                loss_i = np.sum((y_i.reshape((1,-1)) - a_values_i[-1].reshape(-1)[:-(M - NEURONS_INFO[last_layer_idx])])**2)
                loss+=loss_i
                
                dC_dz+=dC_dz_i
                dC_dw+=dC_dw_i
                dC_db+=dC_db_i

        
        # Average loss and gradients
        loss = loss/training_size
        for idx in range(N_LAYERS):
                dC_dz[idx] = np.mean(dC_dz[idx])
                dC_dw[idx] = np.mean(dC_dw[idx])
                dC_db[idx] = np.mean(dC_db[idx])

        
        # Update weights and biases
        for idx in range(N_LAYERS):
            weights_matrices[idx] = weights_matrices[idx] - eta * dC_dw[idx]
            bias_matrices[idx] = bias_matrices[idx] - eta * dC_db[idx]
                
        # Logging
        if (epoch+1) % 5 == 0:
            print(f"===============Epoch {epoch+1}/{epochs}===============")
            print(f"Loss: {loss}\n")
            
    return weights_matrices, bias_matrices
            

Training the network

In [148]:
trained_weights_matrices, trained_bias_matrices = train(X_train, y_train)

===============Epoch 5/10===============
Loss: 0.9267534760903614

===============Epoch 10/10===============
Loss: 0.9267534760903614



Predictions on test data 

In [149]:
y_pred = np.zeros(y_test.shape)
test_size = y_pred.shape[0]

In [150]:
for i in range(test_size):
    _, activated_outputs = forward_pass(X_test[i], trained_weights_matrices, trained_bias_matrices)
    pred = np.asarray(activated_outputs[-1]).flatten()
    pred = pred[:-(M - NEURONS_INFO[N_LAYERS - 1 ])]
    index = np.argmax(pred)
    one_hot = np.zeros(pred.shape, dtype=int)
    one_hot[index] = 1
    y_pred[i] = one_hot
    

In [151]:
y_pred.shape, y_test.shape

((30, 3), (30, 3))

Accuracy

In [152]:
np.sum(np.all(y_pred == y_test, axis=1))/test_size

np.float64(0.3333333333333333)